# Dirac in python: Higher level handling of calculations
_Fernando Mata Rabadán, Université de Lille/KU Leuven, 2026_ 

In this quickguide, we will show the interface between python (ASE) and DIRAC, in the most simple way possible.

## Background
Before starting the calculations, it is necessary to visit very briefly how `ase` works. Molecules are defined in `ase.Atoms` objects, which presents numerous properties and methods. Associated to an `Atoms` object, there is a calculator `Atoms.calc` (`FileIOCalculator`), which handles the input writing, execution and data extraction to pass to the `Atoms` class. In our case it is the `DIRAC(FileIOCalculator)` class. Therefore, when any property is requested to the `Atoms` object, it will call the `DIRAC.run_calculation_method()`. 

This guide shows how to set up and use the `DIRAC` calculator, and how to utilize different methods for:
- Input generation
- Analyzing results
- Visualization


In [ ]:
from ase import Atoms
from ase.units import Ha
from ase_dirac import DIRAC
from pathlib import Path
import shutil

notebook_path = Path().parent.resolve()

h2_path =  f"{str(notebook_path)}/geometries/h2.xyz"
h2o_path = f"{str(notebook_path)}/geometries/h2o.xyz"


## 1. Preparing and running calculations
The simplest way to perform a calculation is to provide a geometry, and build an `Atoms` obhect. For this there are four options: 
1. Just input geometry: A calculator with default SCF behaviour will be used. 
2. Input geometry and a input file: A calculator based on the input file will be used.
3. Input geometry and a pass the inout as a string: A calculator based on the string will be used.
4. Input geometry and a template: A calculator based on the chosen template will be used.

In [ ]:
DiracAtoms: Atoms = DIRAC.setup_geom_and_imp(h2_path)
DiracAtoms.get_potential_energy() / Ha

In [ ]:
shutil.copyfile('SCF.inp', 'SCF2.inp')
DiracAtoms = DIRAC.setup_geom_and_imp(h2_path, inp_filename="SCF2.inp")
DiracAtoms.get_potential_energy() / Ha

In [ ]:
with open("SCF.inp") as f:
    inp_string = f.read() 
DiracAtoms = DIRAC.setup_geom_and_imp(h2_path, inp_string=inp_string)
DiracAtoms.get_potential_energy() / Ha

In [ ]:
DiracAtoms = DIRAC.setup_geom_and_imp(h2_path, template_input_str="CCSD")
DiracAtoms.get_potential_energy() / Ha


---

## 2. Modifying sections
But what if I don't like the content of the sections? They can be easily modified with the methods: 
- `DIRAC.override_section`: Overrides the content of a section with the inputed dirac-formated string.
- `DIRAC.add_section`: Adds a section using a keyword, then it can be overriden with the previous method.
- `DIRAC.remove_section`: Removes entirely a section. 

We will start with the previous example, we are building from the SCF template of H2, but we will now do the following: 
- Perform a DFT calculation instead of using the `.LVCORR` Hamiltonian. 
- Change occupation.
- Change the basis set. 

In [ ]:
DiracAtomsMod = DIRAC.setup_geom_and_imp(h2_path, template_input_str="SCF")

print("Before modification:\n")
DiracAtomsMod.calc.print_parameters()

DiracAtoms.calc.override_section("""**MOLECULE
*BASIS
.DEFAULT
cc-pvqz""")

DiracAtomsMod.calc.override_section("""
**HAMILTONIAN
.DFT
B3LYP
""")

DiracAtomsMod.calc.override_section("""
**WAVE FUNCTIONS
.SCF
*SCF
.CLOSED SHELL
2 2 
""")

print("\nAfter modification:\n")
DiracAtomsMod.calc.print_parameters()

In [ ]:
DiracAtomsMod.get_potential_energy() / Ha

Or if we want to modify the wave function section to do CCSD too:

In [ ]:
DiracAtomsMod = DIRAC.setup_geom_and_imp(h2_path, template_input_str="SCF")

print("Before modification:\n")
DiracAtomsMod.calc.print_parameters()

DiracAtomsMod.calc.override_section("""
    **WAVE FUNCTIONS
    .SCF
    .RELCCSD
    *SCF
    .PRINT 
    1
    **RELCCSD
    .ENERGY
    .PRINT
    1
""")

print("Before modification:\n")
DiracAtomsMod.calc.print_parameters()
print('\n')

DiracAtomsMod.get_potential_energy() / Ha


---


## After the calculation is over: retrieving the info
The results of the calculation are stored in the `DIRAC` object, and can be retrieved with the following methods:

In [ ]:
DiracAtomsMod.calc.results.keys()

### `.h5` functionality from the calculator 
Under the hood what is happening is that the `read_results()` method in the calculator was extended to retrieve some data from the `CHECKPOINT.h5` file generated in the calculation. We can see the content as:

In [ ]:
DiracAtomsMod.calc.list_h5_contents[0:15]

And extract a property as:

In [ ]:
n_ao = DiracAtomsMod.calc.extract_h5_value("input/aobasis/1/n_ao")
print(*n_ao)


---


# Visualization
It is possible to request visual properties. This is performed in two steps (here). First we set up and run the calculation as:

In [ ]:
DiracAtomsVis = DIRAC.setup_geom_and_imp(h2o_path, template_input_str="SCF")
DiracAtomsVis.get_potential_energy() / Ha

Then we can request the density as:

In [ ]:
DiracAtomsVis.calc.request_visualization_density()

Or input directly the `.GRIDS` and `.GRIDFUCNTIONS`: 

In [ ]:
DiracAtomsVis.calc.add_visualization_property(
    "id=2 input=create typ=1 npoints=[64,64,32] margin=2.0",
    "name=elf id_grid=2 purpose=visualization export=cube file_out=SCF_elf.cube",
)

DiracAtomsVis.calc.add_visualization_property(
    "id=3 input=dft",
    "name=ed id_grid=3 purpose=integration",
)

DiracAtomsVis.calc.parameters

Then we compute the visual properties with:

In [ ]:
DiracAtomsVis.calc.compute_visual_properties()

And finally we can visualize the results interactively:

In [ ]:
DiracAtomsVis.calc.plot_geom()

See the list of availiable `.cube` files:

In [ ]:
print(f'Availiable cube files (DiracAtomsVis.calc.list_cubes): {DiracAtomsVis.calc.list_cubes}' )

And plot them with the method:

In [ ]:
DiracAtomsVis.calc.plot_property('SCF_ed.cube')

In [ ]:
view = DiracAtomsVis.calc.plot_property('SCF_elf.cube',  isovalue=0.9)